# Notebook 2 â€” Embed Cleaned Maryland Polyvore Dataset

Runs every `top.jpg` and `bottom.jpg` through the FashionCLIP INT8 ONNX model
and saves three aligned numpy arrays for use in training.

**Before running:**
1. Upload `fashion_clip_vision_int8.onnx` (from Notebook 1) as a Kaggle dataset input
2. Upload `Cleaned-Maryland-Dataset/` folder as a Kaggle dataset input

**Output (save as Kaggle dataset output):**
- `outfit_ids.npy` â€” outfit folder names, shape (N,)
- `top_embeddings.npy` â€” shape (N, 512) float32
- `bottom_embeddings.npy` â€” shape (N, 512) float32

All arrays are aligned: index i in each array refers to the same outfit.

**Run on:** Kaggle (CPU is fine â€” ONNX INT8 is fast enough)

In [ ]:
!pip install -q onnxruntime Pillow numpy tqdm torchvision torch

In [ ]:
import os
from pathlib import Path

DATASET_DIR = "/kaggle/input/cleaned-maryland-polyvore/Cleaned-Maryland-Dataset"
OUTPUT_DIR  = "/kaggle/working"
BATCH_SIZE  = 64  # reduce to 32 if memory issues

# Auto-find the ONNX model -- works whether uploaded as dataset or model
_matches = list(Path("/kaggle/input").rglob("fashion_clip_vision_int8.onnx"))
assert _matches, f"fashion_clip_vision_int8.onnx not found. Available: {os.listdir('/kaggle/input')}"
MODEL_PATH = str(_matches[0])
print(f"Model: {MODEL_PATH}")

assert os.path.isdir(DATASET_DIR), (
    f"Dataset not found: {DATASET_DIR}"
    f" | Available: {os.listdir('/kaggle/input')}"
)
print("Paths OK")


In [ ]:
import numpy as np
from PIL import Image
import onnxruntime as ort
from tqdm import tqdm
from pathlib import Path
from torchvision import transforms

# Preprocessing pipeline â€” must exactly match what open_clip uses for ViT-B/32.
# These are the standard CLIP image normalisation constants.
preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),  # converts PIL [0,255] to float32 [0,1] and (H,W,C) -> (C,H,W)
    transforms.Normalize(
        mean=[0.48145466, 0.4578275,  0.40821073],
        std= [0.26862954, 0.26130258, 0.27577711],
    ),
])

def preprocess_to_numpy(path: str) -> np.ndarray:
    """Load one image and return (1, 3, 224, 224) float32 numpy array."""
    img    = Image.open(path).convert("RGB")
    tensor = preprocess(img)            # (3, 224, 224)
    return tensor.unsqueeze(0).numpy()  # (1, 3, 224, 224)

def preprocess_batch(paths) -> np.ndarray:
    """Load and preprocess a list of image paths. Returns (B, 3, 224, 224)."""
    import torch
    tensors = [preprocess(Image.open(p).convert("RGB")) for p in paths]
    return torch.stack(tensors).numpy()  # (B, 3, 224, 224)

print("Preprocessing pipeline ready")

In [ ]:
# â”€â”€ Discover valid outfit folders â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dataset_path = Path(DATASET_DIR)

outfit_dirs = sorted([
    d for d in dataset_path.iterdir()
    if d.is_dir()
    and (d / "top.jpg").exists()
    and (d / "bottom.jpg").exists()
])

print(f"Valid outfit folders: {len(outfit_dirs):,}")
print(f"Sample folder names:  {[d.name for d in outfit_dirs[:3]]}")

In [ ]:
# â”€â”€ Load ONNX model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
opts = ort.SessionOptions()
opts.inter_op_num_threads = 4
opts.intra_op_num_threads = 4

sess = ort.InferenceSession(MODEL_PATH, sess_options=opts, providers=["CPUExecutionProvider"])

print(f"Input:  {sess.get_inputs()[0].name}  {sess.get_inputs()[0].shape}")
print(f"Output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}")

def embed_batch(paths):
    batch = preprocess_batch(paths)  # (B, 3, 224, 224)
    return sess.run(["embedding"], {"pixel_values": batch})[0]  # (B, 512)

In [ ]:
# â”€â”€ Batch embed all outfits â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
outfit_ids        = []
top_embeddings    = []
bottom_embeddings = []
failed            = []

for i in tqdm(range(0, len(outfit_dirs), BATCH_SIZE), desc="Embedding outfits"):
    batch_dirs   = outfit_dirs[i : i + BATCH_SIZE]
    top_paths    = [str(d / "top.jpg")    for d in batch_dirs]
    bottom_paths = [str(d / "bottom.jpg") for d in batch_dirs]

    try:
        top_embs    = embed_batch(top_paths)
        bottom_embs = embed_batch(bottom_paths)
        valid_dirs  = batch_dirs
    except Exception:
        # Fall back one-by-one to isolate corrupt images
        top_embs, bottom_embs, valid_dirs = [], [], []
        for d in batch_dirs:
            try:
                t = embed_batch([str(d / "top.jpg")])[0]
                b = embed_batch([str(d / "bottom.jpg")])[0]
                top_embs.append(t)
                bottom_embs.append(b)
                valid_dirs.append(d)
            except Exception:
                failed.append(d.name)
        if not top_embs:
            continue
        top_embs    = np.stack(top_embs)
        bottom_embs = np.stack(bottom_embs)

    for j, d in enumerate(valid_dirs):
        outfit_ids.append(d.name)
        top_embeddings.append(top_embs[j])
        bottom_embeddings.append(bottom_embs[j])

print(f"\nEmbedded: {len(outfit_ids):,} outfits")
if failed:
    print(f"Skipped {len(failed)} corrupt outfits")

In [ ]:
# â”€â”€ Save aligned arrays â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
outfit_ids_arr        = np.array(outfit_ids)
top_embeddings_arr    = np.stack(top_embeddings).astype(np.float32)
bottom_embeddings_arr = np.stack(bottom_embeddings).astype(np.float32)

print(f"outfit_ids:        {outfit_ids_arr.shape}")
print(f"top_embeddings:    {top_embeddings_arr.shape}")
print(f"bottom_embeddings: {bottom_embeddings_arr.shape}")

np.save(f"{OUTPUT_DIR}/outfit_ids.npy",        outfit_ids_arr)
np.save(f"{OUTPUT_DIR}/top_embeddings.npy",    top_embeddings_arr)
np.save(f"{OUTPUT_DIR}/bottom_embeddings.npy", bottom_embeddings_arr)

print("Saved. Go to Kaggle â†’ Data â†’ Output â†’ Save Version to persist these files.")

In [ ]:
# â”€â”€ Sanity check â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Same-outfit pairs should have higher cosine similarity than random pairs.
# If they don't, preprocessing does not match what the model expects.

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

n_check = min(500, len(outfit_ids_arr))

same_sims = [
    cosine_sim(top_embeddings_arr[i], bottom_embeddings_arr[i])
    for i in range(n_check)
]

rng = np.random.default_rng(42)
rand_i = rng.integers(0, len(outfit_ids_arr), n_check)
rand_j = rng.integers(0, len(outfit_ids_arr), n_check)
rand_sims = [
    cosine_sim(top_embeddings_arr[i], bottom_embeddings_arr[j])
    for i, j in zip(rand_i, rand_j) if i != j
]

print(f"Same-outfit cosine similarity:  mean={np.mean(same_sims):.3f}  std={np.std(same_sims):.3f}")
print(f"Random-pair cosine similarity:  mean={np.mean(rand_sims):.3f}  std={np.std(rand_sims):.3f}")
print()
if np.mean(same_sims) > np.mean(rand_sims):
    print("Same-outfit similarity is higher than random â€” embeddings look correct.")
else:
    print("WARNING: same-outfit similarity is NOT higher than random.")
    print("Check that MODEL_PATH points to the correct ONNX file from Notebook 1.")

In [ ]:
# ── Package outputs and download ─────────────────────────────────────────────
import zipfile, os

zip_path = f"{OUTPUT_DIR}/polyvore_embeddings.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["outfit_ids.npy", "top_embeddings.npy", "bottom_embeddings.npy"]:
        fpath = f"{OUTPUT_DIR}/{fname}"
        zf.write(fpath, fname)
        print(f"  Added {fname} ({os.path.getsize(fpath) / 1e6:.1f} MB)")

print(f"
Zip: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")
print("
── Next steps ────────────────────────────────────────────────────────")
print("1. Kaggle sidebar → Data → Output → Save Version")
print("   (this persists /kaggle/working/ as a versioned dataset)")
print("2. Add this notebook output as input to train_compatibility_mlp.ipynb")
print("   Dataset slug: polyvore-embeddings  →  /kaggle/input/polyvore-embeddings/")
print("3. Or download polyvore_embeddings.zip from the Output tab directly")
